# Conversational AI — aka Chatbot

**Reference guide:** build a Gradio chat UI, wire it to OpenAI, then shape behavior with system prompts.

### What you'll learn
1. **Gradio `ChatInterface`** — callback signature `chat(message, history)`
2. **Conversation history** — roles (`system` / `user` / `assistant`) and message lists
3. **Streaming** — `yield` partial replies so the UI updates token-by-token
4. **Prompt engineering** — static system messages, one-shot examples, and *dynamic* prompts based on user input

> Run cells top-to-bottom. Each `launch()` opens a new local Gradio app (ports increment: 7860, 7861, …).

In [1]:
# --- Imports ---
# os          : read environment variables (e.g. OPENAI_API_KEY)
# load_dotenv : load secrets from a local .env file into the process env
# OpenAI      : official client for Chat Completions API
# gradio (gr) : quick web UI; ChatInterface gives a ready-made chat layout
import os
from typing import Iterator, TypedDict

from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr


class ChatMessage(TypedDict):
    """One turn in the OpenAI / Gradio messages format."""
    role: str      # "system" | "user" | "assistant"
    content: str


# Load .env so OPENAI_API_KEY is available (override=True refreshes if already set)
load_dotenv(override=True)

# Optional sanity check — confirms the key loaded without printing the full secret
openai_api_key: str | None = os.getenv("OPENAI_API_KEY")
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set — add OPENAI_API_KEY to your .env file")


OpenAI API Key exists and begins sk-proj-


## 1. Initialize the OpenAI client

`OpenAI()` reads `OPENAI_API_KEY` from the environment.  
`MODEL` is a global so you can swap models without rewriting every call.

In [2]:
# Client instance — reuse this object for all API calls in the notebook
openai: OpenAI = OpenAI()

# Model id used by chat.completions.create(...). Smaller/faster models are fine for demos.
MODEL: str = "gpt-4o-mini"


## 2. System message (persona / instructions)

The **system** role tells the model *how* to behave. It is not shown to the user in the chat UI, but it is prepended to every API request.

We'll mutate this global later to change store-assistant behavior without rewriting the chat function.

In [3]:
# Baseline instructions — kept intentionally simple at first
system_message: str = "You are a helpful assistant."


## 3. Gradio callback contract: `chat(message, history)`

`gr.ChatInterface(fn=chat, type="messages")` expects a callback shaped like:

```python
def chat(message: str, history: list[ChatMessage]) -> str:  # or Iterator[str] when streaming
    ...
```

Arguments:

| Argument | Meaning |
|----------|---------|
| `message` | The latest user string |
| `history` | Prior turns as a list of dicts: `{"role": "user"|"assistant", "content": "..."}` |

**Return** a string (or `yield` strings when streaming). Gradio appends your return value as the assistant reply.

### Stub first
Ignore the LLM and return a fixed string so you can verify the UI wiring.


In [4]:
# Stub callback: Gradio will call this on every user send.
# Ignoring message/history proves the UI works before involving the API.
def chat(message: str, history: list[ChatMessage]) -> str:
    return "bananas"


In [5]:
# ChatInterface wires the callback to a browser chat UI.
# type="messages" → history is a list of {"role", "content"} dicts (OpenAI-style).
# .launch() starts a local server (default http://127.0.0.1:7860).
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7873
* To create a public link, set `share=True` in `launch()`.


### Inspect what Gradio passes in

Redefine `chat` to echo `message` (and optionally print `history`) so you can see the callback arguments. Re-run the `launch()` cell above (or the next launch) after redefining the function — Gradio uses whatever `chat` is currently bound to when the interface was created; **re-launch** after changing `chat`.

In [6]:
# Still a stub — but now we surface the inputs so you can inspect them in the UI.
# Tip: print(history) in a notebook cell or temporarily add a print() to debug structure.
def chat(message: str, history: list[ChatMessage]) -> str:
    return f"You said {message} and history is {history} but I still say bananas"


In [7]:
gr.ChatInterface(fn = chat, type = "messages").launch()

* Running on local URL:  http://127.0.0.1:7874
* To create a public link, set `share=True` in `launch()`.


## 4. Real chat: build the OpenAI `messages` list

Pattern every conversational app uses:

```
[ system ] + [ prior turns from history ] + [ latest user message ]
```

Then call `openai.chat.completions.create(...)` and return `response.choices[0].message.content`.

**History cleanup:** Gradio may include extra keys; we project to `{role, content}` only so the API payload stays clean.

In [9]:
def chat(message: str, history: list[ChatMessage]) -> str | None:
    # Keep only role + content (Gradio message objects may carry extra metadata)
    history = [
        ChatMessage(role=h["role"], content=h["content"])
        for h in history
    ]

    # Full prompt stack sent to the model:
    #   1) system instructions
    #   2) prior conversation (user/assistant pairs)
    #   3) the new user turn
    messages: list[ChatMessage] = (
        [ChatMessage(role="system", content=system_message)]
        + history
        + [ChatMessage(role="user", content=message)]
    )

    # Blocking (non-streaming) call — wait for the full reply, then return once
    # content can be None per the API types, so the return type is str | None
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content


In [10]:
# Launch with the real OpenAI-backed callback (non-streaming)
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


## 5. Streaming responses

With `stream=True`, the API yields **chunks**. Gradio treats a generator (`yield`) as a streaming reply: each yield replaces the assistant bubble with the growing text.

Key details:
- Accumulate tokens into `response`, then `yield response` (full text so far — Gradio expects the complete string each time, not just the delta).
- `chunk.choices[0].delta.content` can be `None` on some chunks → use `or ''` to avoid `TypeError`.
- Prefer `yield` over `return` when streaming.

In [11]:
def chat(message: str, history: list[ChatMessage]) -> Iterator[str]:
    # Same history projection as the non-streaming version
    history = [
        ChatMessage(role=h["role"], content=h["content"])
        for h in history
    ]

    messages: list[ChatMessage] = (
        [ChatMessage(role="system", content=system_message)]
        + history
        + [ChatMessage(role="user", content=message)]
    )

    # stream=True → iterable of partial completion chunks
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=messages,
        stream=True,
    )

    response: str = ""  # running concatenation of tokens
    for chunk in stream:
        # delta.content is the new token(s); None on empty/keepalive chunks
        response += chunk.choices[0].delta.content or ""
        yield response  # Gradio redraws the assistant message with the full string so far


In [12]:
# Same UI — now powered by the streaming generator above
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


## 6. Shape behavior with the system prompt (one-shot)

Without changing `chat`, rewrite `system_message` to set:
- **Role** — clothes-store assistant  
- **Goal** — gently push sale items  
- **Facts** — hat/other discounts  
- **Example reply** — one-shot prompting so the model mirrors tone and structure  

Because `chat` reads the global `system_message` on every request, re-launching (or already using the latest global) is enough.

In [13]:
# Richer system prompt: persona + sales goals + concrete discount facts + example reply.
# The example is "one-shot prompting" — show the style you want once, then the model generalizes.
system_message: str = """
You are a helpful assistant in a clothes store. You should try
to gently encourage the customer to try items that are on sale.
Hats are 60% off, and most other items are 50% off. For example,
if the customer says "I'm looking to buy a hat", you could
reply something like, "Wonderful - we have lots of hats -
including several that are part of our sales event."
"""


In [14]:
# Reuses the streaming `chat` from above — only the system_message global changed
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


### Extend the prompt incrementally

Appending rules with `+=` is handy while iterating. Later you may prefer one consolidated prompt string for clarity and version control.

In [15]:
# Add a product-specific rule without rewriting the whole prompt.
# Note: this mutates the global — re-run earlier cells carefully if you reset the notebook.
system_message += (
    "\nIf the customer asks for shoes, you should respond that shoes are not on sale today, "
    "but remind the customer to look at hats!"
) 

In [17]:
# Try asking about shoes — the model should deflect to hats per the appended rule
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


## 7. Dynamic system prompts (per-message)

Sometimes you only want extra instructions when the user mentions a topic (saves tokens; keeps the default prompt lean).

Here: if `"belt"` appears in the user message, append a “we don’t sell belts” rule for **that request only**. Other turns keep the base `system_message`.

This is a lightweight form of **conditional / retrieval-style prompting** — scale this idea up with keyword routing, classifiers, or RAG.

In [18]:
def chat(message: str, history: list[ChatMessage]) -> Iterator[str]:
    history = [
        ChatMessage(role=h["role"], content=h["content"])
        for h in history
    ]

    # Start from the shared store prompt; optionally specialize for this turn
    relevant_system_message: str = system_message

    # Case-insensitive keyword gate — only inject belt rules when needed
    if "belt" in message.lower():
        relevant_system_message += (
            " The store does not sell belts; if you are asked for belts, "
            "be sure to point out other items on sale."
        )

    messages: list[ChatMessage] = (
        [ChatMessage(role="system", content=relevant_system_message)]
        + history
        + [ChatMessage(role="user", content=message)]
    )

    stream = openai.chat.completions.create(
        model=MODEL,
        messages=messages,
        stream=True,
    )

    response: str = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        yield response


In [19]:
# Final demo: ask about belts vs hats/shoes to see conditional prompting in action
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


## Quick reference (cheat sheet)

| Piece | Role |
|-------|------|
| `load_dotenv` + `OPENAI_API_KEY` | Auth for the OpenAI client |
| `system_message` | Persistent persona / business rules |
| `ChatMessage` | `TypedDict` with `role` + `content` |
| `chat(message, history)` | Gradio callback; builds the `messages` list |
| `-> str` / `-> Iterator[str]` | Non-streaming vs streaming return types |
| `history` projection | Strip Gradio extras → `{role, content}` |
| `messages = [system] + history + [user]` | Standard multi-turn chat payload |
| `return content` | Non-streaming full reply |
| `yield growing_string` | Streaming UI updates |
| `system_message += ...` | Iterate on static rules |
| Keyword / condition on `message` | Dynamic per-turn instructions |

### Business takeaway
Conversational assistants are a common Gen AI use case. Gradio gets you a UI fast; prompting supplies context, tone, and examples. Prototype your own domain by swapping the system prompt for your business facts and voice.
